In [2]:
import os, re, numpy as np, nltk, kagglehub
from nltk.tokenize import sent_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras import Sequential, layers, callbacks
from sklearn.model_selection import train_test_split

nltk.download('punkt_tab')
path = kagglehub.dataset_download("ronikdedhia/next-word-prediction")
raw = open(os.path.join(path, os.listdir(path)[0]), encoding='utf-8', errors='ignore').read().lower()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Using Colab cache for faster access to the 'next-word-prediction' dataset.


In [3]:
clean = lambda s: " ".join(re.sub(r"[^a-z\s]", " ", s.replace("'", "").replace("’", "")).split())
sentences = [c for c in (clean(s) for s in sent_tokenize(raw)) if len(c.split()) >= 3]

In [4]:
tok = Tokenizer(oov_token='<unk>')
tok.fit_on_texts(sentences)
V = len(tok.word_index) + 1

In [5]:
grams = [s[:i] for s in tok.texts_to_sequences(sentences) for i in range(2, len(s) + 1)]
MAXLEN = min(max(map(len, grams)), 40)
grams = pad_sequences(grams, maxlen=MAXLEN, padding='pre')
X, y = grams[:, :-1], grams[:, -1]
print(f"vocab={V}, samples={len(X)}, input_len={X.shape[1]}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, random_state=42)

vocab=8174, samples=103601, input_len=39


In [6]:
model = Sequential([
    layers.Input(shape=(MAXLEN - 1,)),
    layers.Embedding(V, 128),
    layers.LSTM(256, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(256),
    layers.Dense(V, activation='softmax'),
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=40, batch_size=256,
          callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

Epoch 1/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 638s 2s/step - accuracy: 0.0529 - loss: 6.8028 - val_accuracy: 0.0560 - val_loss: 6.6075
Epoch 2/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 677s 2s/step - accuracy: 0.0640 - loss: 6.3267 - val_accuracy: 0.0768 - val_loss: 6.2228
Epoch 3/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 685s 2s/step - accuracy: 0.0905 - loss: 5.9477 - val_accuracy: 0.1019 - val_loss: 5.9636
Epoch 4/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 651s 2s/step - accuracy: 0.1097 - loss: 5.6631 - val_accuracy: 0.1149 - val_loss: 5.8618
Epoch 5/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.1231 - loss: 5.4547 - val_accuracy: 0.1260 - val_loss: 5.7938
Epoch 6/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 634s 2s/step - accuracy: 0.1342 - loss: 5.2842 - val_accuracy: 0.1324 - val_loss: 5.7655
Epoch 7/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 696s 2s/step - accuracy: 0.1429 - loss: 5.1376 - val_accuracy: 0.1314 - val_loss: 5.7475
Epoch 8/40
365/365 ━━━━━━━━━━━━━━━━━━━━ 649s 2s/step - accuracy: 0.1477 - loss: 5.0057 - val_accu

In [7]:
def predict_next(text, k=3):
    seq = tok.texts_to_sequences([clean(text.lower())])[0][-(MAXLEN - 1):]
    seq = pad_sequences([seq], maxlen=MAXLEN - 1, padding='pre')
    p = model.predict(seq, verbose=0)[0]
    p[0] = 0; p[tok.word_index['<unk>']] = 0
    return [(tok.index_word[i], round(float(p[i]), 3)) for i in p.argsort()[-k:][::-1]]

def generate(text, n=5):
    for _ in range(n):
        text += " " + predict_next(text, 1)[0][0]
    return text

In [8]:
for s in ["holmes took up his", "i have no doubt that", "it is a capital mistake to",
          "the man with the twisted", "my dear watson you have"]:
    print(f"{s!r} -> {predict_next(s)}")
    print("   generated:", generate(s))

'holmes took up his' -> [('eyes', 0.057), ('face', 0.043), ('hand', 0.036)]
   generated: holmes took up his eyes and the door and
'i have no doubt that' -> [('i', 0.136), ('he', 0.102), ('you', 0.091)]
   generated: i have no doubt that i have not been a
'it is a capital mistake to' -> [('the', 0.135), ('be', 0.065), ('me', 0.042)]
   generated: it is a capital mistake to the other and and a
'the man with the twisted' -> [('room', 0.023), ('and', 0.021), ('side', 0.012)]
   generated: the man with the twisted room and the door and
'my dear watson you have' -> [('been', 0.142), ('not', 0.035), ('a', 0.034)]
   generated: my dear watson you have been a little man and
